In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:98% !important;}
div.cell.code_cell.rendered{width:98%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li, div.text_cell_render p{width:95% !important;font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
table td, th{font-size:16px;}
table{ margin-left:0 !important;   /* 왼쪽 여백 0 */}
</style>
"""))

# 1. knowledge base 구성을 위한 데이터 생성

In [6]:
%%time
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
loader = Docx2txtLoader("tax_docx/소득세법_with_table.docx")
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,   # 문서를 쪼갤 때 1500글자씩 chunking
    chunk_overlap=20, # 20문자씩 overlap 하여 chunking
    # separators=["\n\n", "\n", " ", ""]
)
# Recursive(재귀적으로) 다음의 순서대로 chunking 시도 : "\n\n", "\n", " ", ""
documents = loader.load_and_split(text_splitter=text_splitter)
len(documents)

CPU times: total: 3.94 s
Wall time: 3.91 s


180

In [7]:
documents[43]

Document(metadata={'source': 'tax_docx/소득세법_with_table.docx'}, page_content='제54조의2(공동사업에 대한 소득공제 등 특례) 제51조의3 또는 「조세특례제한법」에 따른 소득공제를 적용하거나 제59조의3에 따른 세액공제를 적용하는 경우 제43조제3항에 따라 소득금액이 주된 공동사업자의 소득금액에 합산과세되는 특수관계인이 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액이 있으면 주된 공동사업자의 소득에 합산과세되는 소득금액의 한도에서 주된 공동사업자가 지출ㆍ납입ㆍ투자ㆍ출자 등을 한 금액으로 보아 주된 공동사업자의 합산과세되는 종합소득금액 또는 종합소득산출세액을 계산할 때에 소득공제 또는 세액공제를 받을 수 있다. <개정 2012. 1. 1., 2014. 1. 1.>\n\n[전문개정 2009. 12. 31.]\n\n[제목개정 2014. 1. 1.]\n\n\n\n제4절 세액의 계산 <개정 2009. 12. 31.>\n\n\n\n제1관 세율 <개정 2009. 12. 31.>\n\n\n\n제55조(세율) ①거주자의 종합소득에 대한 소득세는 해당 연도의 종합소득과세표준에 다음의 세율을 적용하여 계산한 금액(이하 “종합소득산출세액”이라 한다)을 그 세액으로 한다. <개정 2014. 1. 1., 2016. 12. 20., 2017. 12. 19., 2020. 12. 29., 2022. 12. 31.>\n\n\n\n\n\n\n\n\n\n종합소득 과세표준\n\n세율\n\n1,400만원 이하\n\n과세표준의 6퍼센트\n\n1,400만원 초과 5,000만원 이하\n\n84만원 + (1,400만원을 초과하는 금액의 15퍼센트)\n\n5,000만원 초과 8,800만원 이하\n\n624만원 + (5,000만원을 초과하는 금액의 24퍼센트)\n\n8,800만원 초과 1억 5천만원 이하\n\n1,536만원 + (8,800만원을 초과하는 금액의 35퍼센트)\n\n1억5천만원 초과 3억원 이하\n\n3,706만원 + (1억5천만원을 초과하는 금액의

In [11]:
from dotenv import load_dotenv
from langchain_upstage import UpstageEmbeddings
load_dotenv(override=True) # 커널을 시작하지 않고 .env덮어쓰기
embedding = UpstageEmbeddings(
                model = 'embedding-query'
)

In [10]:
len(embedding.embed_query("소득세법"))

4096

In [24]:
# pinecone의 index 생성하기
import time
from pinecone import Pinecone, ServerlessSpec
# 0. pinecone 객체 생성
pc = Pinecone() # api_key=os.getenv('PINECONE_API_KEY')

# 1. 인덱스 설정
index_name = 'tax-index-table'
dimension = len(embedding.embed_query("차원 확인용"))

# 2. 인덱스 생성(해당 인덱스가 없을 경우)
existing_indexes = [index.name for index in pc.list_indexes()]
# print(index_name in existing_indexes)
if index_name not in existing_indexes:
    pc.create_index(
        name=index_name, 
        dimension=dimension,
        metric='cosine',  # 유사도 측정 방식
        spec=ServerlessSpec(
            cloud='aws',    # 무료 플랜은 aws
            region='us-east-1' # 무료플랜은 us-east-1만 가능
        )        
    )
    # 인덱스가 준비될 때까지 대기
    while not pc.describe_index(index_name).status['ready']:
        time.sleep(1) # 1초 대기
    print(f'인덱스 {index_name}를 생성하였습니다')
else:
    print(f'인덱스 {index_name}이 이미 존재합니다')

인덱스 tax-index-table이 이미 존재합니다


In [28]:
# 3. 인덱스 정보 확인
print([index.name for index in pc.list_indexes()])
print(pc.describe_index(index_name).status['ready'])
index = pc.Index(index_name)
print(index.describe_index_stats())

['tax-index-openai', 'tax-index-table', 'tax-index']
True
{'dimension': 4096,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {},
 'total_vector_count': 0,
 'vector_type': 'dense'}


In [29]:
%%time
# 4. 문서를 임베딩해서 클라우드 vector DB(index)에 저장
# 처음 한 번
from langchain_pinecone import PineconeVectorStore
# database = PineconeVectorStore.from_documents(
#     documents=documents,
#     embedding=embedding,
#     index_name=index_name
# )
# 업로드한 벡터DB를 load할 때
database = PineconeVectorStore(
    embedding=embedding,
    index_name=index_name
)

CPU times: total: 6.84 s
Wall time: 27.7 s
